# Support Vector Classifiers

Here we try many support vector classifiers, with different kernels. 

In [ ]:
import pandas as pd

learn_data = pd.read_csv("../data/minimal_train_fs.csv", header = None)
learn_data.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female', 'Target']
learn_data["Female"] = learn_data["Female"].astype("category")
learn_data["Target"] = learn_data["Target"].astype("category")
learn_data.head()

,Age,TB,Alkphos,Sgot,ALB,AR,BilRatio,Female,Target
0,48,1.504077,5.641907,4.304065,2.4,0.52,0.511111,0,0
1,39,0.641854,5.192957,4.127134,4.3,1.38,0.473684,0,0
2,23,0.000000,5.356586,4.382027,3.1,1.00,0.300000,0,0
3,42,-0.356675,5.023881,4.394449,3.2,1.06,0.285714,1,0
4,54,3.117950,6.324359,3.610918,3.4,0.80,0.504425,1,0


In [2]:
from sklearn.model_selection import train_test_split

X = learn_data.drop(columns = ["Target"])
Xnum = X.drop(columns = ["Female"])
y = learn_data["Target"]

X_train, X_val, Xnum_train, Xnum_val, y_train, y_val = train_test_split(X, Xnum, y, test_size = 0.33, random_state = 42)

## Metrics

In [3]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import numpy as np

def compute_metrics (y_real, y_pred) -> list[float]:
    F1_macro = f1_score(y_real, y_pred, average = "macro")
    recall = recall_score(y_real, y_pred, average = "macro")
    prec = precision_score(y_real, y_pred, average = "macro")
    acc = accuracy_score(y_real, y_pred)
    return [F1_macro, recall, prec, acc]

def confusion (y_real, y_pred) -> None:
    TP = sum(np.logical_and(y_real == y_pred, y_real == 1))
    TN = sum(np.logical_and(y_real == y_pred, y_real == 0))
    FP = sum(np.logical_and(y_real != y_pred, y_real == 0))
    FN = sum(np.logical_and(y_real != y_pred, y_real == 1))
    print("\t\tPredicted")
    print("\t\t+1\t0")
    print(f"Real\t+1\t{TP}\t{FN}")
    print(f"\t0\t{FP}\t{TN}")
    print(f"Accuracy: {((TP + TN) / y_real.shape[0] * 100):.2f}%".format())

metrics_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

## Linear kernel (or no kernel)

We have seen with other linear classifiers that the performance is not very good because of two reasons:
- Excessive resampling: we might be resampling too much, and this may affect our predictive power by creating samples that do not correspond to the real data.

In [4]:
from sklearn.svm import LinearSVC

linear_model = LinearSVC(class_weight = "balanced")
linear_model.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(linear_model.predict(Xnum_train)))

		Predicted
		+1	0
Real	+1	73	11
	0	87	129
Accuracy: 67.33%


In [5]:
confusion(np.array(y_val), pd.Series(linear_model.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	32	10
	0	40	67
Accuracy: 66.44%


In [6]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

linear_model = LinearSVC(class_weight = "balanced")
linsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", linear_model)])

n = 100
Cs = np.logspace(start = -1, stop = 2, num = n)

linsvc_search = GridSearchCV(estimator = linsvc_pipeline,
                             param_grid = {'svc__C' : Cs},
                             scoring = 'f1_macro',
                             cv = 5)
linsvc_search.fit(X, y)
linsvc_search.best_params_

{'svc__C': 1.232846739442066}

In [7]:
linsvc_search.best_score_

0.6453852620656173

In [39]:
from sklearn.model_selection import cross_validate

linsvc_C = linsvc_search.best_params_['svc__C']
linsvc_best = LinearSVC(C = linsvc_C, class_weight = "balanced")
linsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", linsvc_best)])

cross_val_results = pd.DataFrame(cross_validate(linsvc_pipeline, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["Linear SVC", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Gaussian SVC,0.660344,0.702813,0.667546,0.68392
Sigmoid SVC,0.655335,0.706255,0.668552,0.674981
Linear SVC,0.652585,0.719721,0.678466,0.666042
Polynomial SVC,0.567243,0.570077,0.606543,0.712684


## Gaussian kernel

In [9]:
from sklearn.svm import SVC

rbf_scale_model = SVC(kernel = "rbf", gamma = "scale", class_weight = "balanced")
rbf_scale_model.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(rbf_scale_model.predict(Xnum_train)))

		Predicted
		+1	0
Real	+1	30	54
	0	40	176
Accuracy: 68.67%


In [10]:
confusion(np.array(y_val), pd.Series(rbf_scale_model.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	14	28
	0	28	79
Accuracy: 62.42%


In [11]:
rbfsvc = SVC(kernel = "rbf", class_weight = "balanced")
rbfsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", rbfsvc)])

n = 50
m = 50
Cs = np.logspace(start = -1, stop = 2, num = n)
gammas = np.logspace(start = -1, stop = 1, num = m) / X.shape[0]

rbfsvc_search = GridSearchCV(estimator = rbfsvc_pipeline,
                             param_grid = {'svc__C' : Cs,
                                           'svc__gamma' : gammas},
                             scoring = 'f1_macro',
                             cv = 5)
rbfsvc_search.fit(X, y)
rbfsvc_search.best_params_

{'svc__C': 1.4563484775012436, 'svc__gamma': 0.0002687734166234585}

In [12]:
rbfsvc_search.best_score_

0.660344012069942

In [13]:
rbfsvc_C = rbfsvc_search.best_params_['svc__C']
rbfsvc_gamma = rbfsvc_search.best_params_['svc__gamma']
rbfsvc_best = SVC(kernel = "rbf", C = rbfsvc_C, gamma = rbfsvc_gamma, class_weight = "balanced")
rbfsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", rbfsvc_best)])

cross_val_results = pd.DataFrame(cross_validate(rbfsvc_pipeline, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["Gaussian SVC", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Gaussian SVC,0.660344,0.702813,0.667546,0.68392
Linear SVC,0.652603,0.719721,0.678289,0.666017


## Polynomial kernel

In [14]:
poly_model = SVC(kernel = "poly", degree = 2, gamma = "scale", class_weight = "balanced")
poly_model.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(poly_model.predict(Xnum_train)))

		Predicted
		+1	0
Real	+1	60	24
	0	118	98
Accuracy: 52.67%


In [15]:
confusion(np.array(y_val), pd.Series(poly_model.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	30	12
	0	53	54
Accuracy: 56.38%


In [33]:
polysvc = SVC(kernel = "poly", class_weight = "balanced")
polysvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", polysvc)])

n = 50
m = 50
Cs = np.logspace(start = -1, stop = 2, num = n)
gammas = np.logspace(start = -2, stop = 2, num = m) / X.shape[0]
degrees = [2, 3]

polysvc_search = GridSearchCV(estimator = polysvc_pipeline,
                              param_grid = {'svc__C' : Cs,
                                            'svc__gamma' : gammas,
                                            'svc__degree' : degrees},
                              scoring = 'f1_macro',
                              cv = 5)
polysvc_search.fit(X, y)
polysvc_search.best_params_

{'svc__C': 2.5595479226995357,
 'svc__degree': 3,
 'svc__gamma': 0.22271714922049}

In [34]:
polysvc_search.best_score_

0.6292882433964854

In [35]:
polysvc_C = polysvc_search.best_params_['svc__C']
polysvc_gamma = polysvc_search.best_params_['svc__gamma']
polysvc_degree = polysvc_search.best_params_['svc__degree']
polysvc_best = SVC(kernel = "poly", C = polysvc_C, gamma = polysvc_gamma, degree = polysvc_degree)
polysvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", polysvc_best)])

cross_val_results = pd.DataFrame(cross_validate(polysvc_pipeline, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["Polynomial SVC", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Gaussian SVC,0.660344,0.702813,0.667546,0.68392
Sigmoid SVC,0.655335,0.706255,0.668552,0.674981
Linear SVC,0.652603,0.719721,0.678289,0.666017
Polynomial SVC,0.567243,0.570077,0.606543,0.712684


## Sigmoid

In [19]:
sig_model = SVC(kernel = "sigmoid", gamma = "scale", class_weight = "balanced")
sig_model.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(sig_model.predict(Xnum_train)))

		Predicted
		+1	0
Real	+1	38	46
	0	100	116
Accuracy: 51.33%


In [20]:
confusion(np.array(y_val), pd.Series(sig_model.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	16	26
	0	55	52
Accuracy: 45.64%


In [21]:
sigsvc = SVC(kernel = "sigmoid", class_weight = "balanced")
sigsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", sigsvc)])

n = 50
m = 50
Cs = np.logspace(start = -1, stop = 2, num = n)
gammas = np.logspace(start = -2, stop = 2, num = m) / X.shape[0]

sigsvc_search = GridSearchCV(estimator = sigsvc_pipeline,
                             param_grid = {'svc__C' : Cs,
                                            'svc__gamma' : gammas},
                             scoring = 'f1_macro',
                             cv = 5)
sigsvc_search.fit(X, y)
sigsvc_search.best_params_

{'svc__C': 1.4563484775012436, 'svc__gamma': 0.0005438871034629513}

In [24]:
sigsvc_search.best_score_

0.6549672865235896

In [26]:
sigsvc_C = sigsvc_search.best_params_['svc__C']
sigsvc_gamma = sigsvc_search.best_params_['svc__gamma']
sigsvc_best = SVC(kernel = "sigmoid", C = sigsvc_C, gamma = sigsvc_gamma, class_weight = "balanced")
sigsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", sigsvc_best)])

cross_val_results = pd.DataFrame(cross_validate(sigsvc_pipeline, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["Sigmoid SVC", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Gaussian SVC,0.660344,0.702813,0.667546,0.68392
Sigmoid SVC,0.655335,0.706255,0.668552,0.674981
Linear SVC,0.652603,0.719721,0.678289,0.666017


## Trying our best models on test dataset


In [ ]:
test_data = pd.read_csv("../data/minimal_test_fs.csv", header = None)
test_data.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female']
test_data["Female"] = learn_data["Female"].astype("category")
test_data.head()

,Age,TB,Alkphos,Sgot,ALB,AR,BilRatio,Female
0,11,-0.356675,6.383507,3.367296,4.2,1.40,0.142857,0
1,62,0.587787,5.411646,5.043425,4.0,0.80,0.500000,0
2,60,-0.356675,5.159055,2.639057,4.2,1.10,0.285714,0
3,60,1.740466,5.365976,6.745236,3.2,0.78,0.491228,1
4,48,-0.105361,5.164786,3.988984,2.7,0.90,0.222222,1


In [28]:
test_data_num = test_data.drop(columns = ["Female"])

### Linear kernel

In [37]:
linsvc_pipeline.fit(Xnum, y)

labels_lin = pd.DataFrame(columns = ['ID', 'Label'])
labels_lin['Label'] = pd.DataFrame(linsvc_pipeline.predict(test_data_num))
labels_lin['ID'] = labels_lin.index + 1
labels_lin

,ID,Label
0,1,1
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,1
112,113,0
113,114,1
114,115,1


### Gaussian kernel (no categorical)

In [30]:
rbfsvc_pipeline.fit(Xnum, y)

labels_rbf = pd.DataFrame(columns = ['ID', 'Label'])
labels_rbf['Label'] = pd.DataFrame(rbfsvc_pipeline.predict(test_data_num))
labels_rbf['ID'] = labels_rbf.index + 1
labels_rbf

,ID,Label
0,1,1
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,1
112,113,0
113,114,1
114,115,1


In [ ]:
labels_rbf.to_csv('../data/new_predictions/rbfsvc_best.csv', index = False)

### Polynomial kernel

In [42]:
polysvc_pipeline.fit(Xnum, y)

labels_poly = pd.DataFrame(columns = ['ID', 'Label'])
labels_poly['Label'] = pd.DataFrame(polysvc_pipeline.predict(test_data_num))
labels_poly['ID'] = labels_rbf.index + 1
labels_poly

,ID,Label
0,1,1
1,2,0
2,3,1
3,4,0
4,5,0
...,...,...
111,112,0
112,113,0
113,114,0
114,115,1


In [ ]:
labels_poly.to_csv('../data/new_predictions/polysvc_best.csv', index = False)

### Sigmoid kernel

In [31]:
sigsvc_pipeline.fit(Xnum, y)

labels_sig = pd.DataFrame(columns = ['ID', 'Label'])
labels_sig['Label'] = pd.DataFrame(sigsvc_pipeline.predict(test_data_num))
labels_sig['ID'] = labels_sig.index + 1
labels_sig

,ID,Label
0,1,1
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,1
112,113,0
113,114,1
114,115,1


In [ ]:
labels_sig.to_csv('../data/new_predictions/sigsvc_best.csv', index = False)

### Discrepancies between models

None at all between gaussian and sigmoid kernels, but a lot of discrepancy with the gaussian

In [36]:
(labels_rbf == labels_sig).value_counts()

ID    Label
True  True     116
Name: count, dtype: int64

In [38]:
(labels_rbf == labels_lin).value_counts()

ID    Label
True  True     100
      False     16
Name: count, dtype: int64